# Train Intent Classifier (tiếng Việt) — E-commerce Support Copilot

Notebook chạy trên **Kaggle**. Dữ liệu: `dataset.jsonl` (1000 câu, 14 nhãn) từ `data/intent_eval/`.

**Chuẩn bị trên Kaggle**
1. *Add Data → Upload* file `dataset.jsonl` (và tuỳ chọn `results_gemini-*.jsonl` để so sánh với LLM).
2. Phần 1–5 (TF-IDF + SVM/LR) chạy được trên CPU.
3. Phần 6 (fine-tune PhoBERT/XLM-R) cần bật **Accelerator: GPU** và **Internet: On**.

**Kết quả xuất ra** `/kaggle/working/`:
- `intent_tfidf_svm.pkl`: sklearn Pipeline có `predict_proba`. Thay vào `artifacts/models/intent_model.pkl` là chạy được với `IntentPredictor` (nhánh single pipeline).
- `intent_transformer/`: model HuggingFace (nếu chạy phần 6).
- `metrics.json`, `test_predictions.csv`.

In [ ]:
import glob, json, os, random, re, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED); np.random.seed(SEED)

def find(pattern):
    hits = glob.glob(f"/kaggle/input/**/{pattern}", recursive=True) + glob.glob(pattern) + glob.glob(f"data/intent_eval/{pattern}")
    return hits

DATA_PATH = find("dataset.jsonl")[0]
OUT_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "./outputs"
os.makedirs(OUT_DIR, exist_ok=True)
print("Data:", DATA_PATH, "| Output:", OUT_DIR)

df = pd.read_json(DATA_PATH, lines=True)
LABELS = ["track_order", "cancel_order", "delivery_time", "delivery_issue", "shipping_costs",
          "return_policy", "return_product", "refund_policy", "request_refund", "refund_status",
          "payment_issue", "human_agent", "other_ecommerce", "out_of_scope"]
assert set(df.intent) <= set(LABELS), set(df.intent) - set(LABELS)
df.head()

## 1. Khám phá dữ liệu

In [ ]:
print(len(df), "câu")
display(df.intent.value_counts().reindex(LABELS).to_frame("n"))
display(df["style"].value_counts().head(15).to_frame("n"))
df["n_words"] = df.text.str.split().str.len()
df.n_words.hist(bins=40); plt.title("Số từ / câu"); plt.show()

## 2. Chia train / test

Chia **phân tầng theo nhãn, TRƯỚC mọi bước xử lý** để không rò rỉ dữ liệu. Test 20% (200 câu) được giữ cố định và dùng chung cho mọi model, kể cả LLM.

In [ ]:
from sklearn.model_selection import train_test_split
train_df, test_df = train_test_split(df, test_size=0.2, stratify=df.intent, random_state=SEED)
train_df, test_df = train_df.reset_index(drop=True), test_df.reset_index(drop=True)
print(len(train_df), "train |", len(test_df), "test")

## 3. Tiền xử lý

Chuẩn hoá nhẹ: hạ chữ thường, gom mã đơn/số thành token chung, rút gọn ký tự lặp ("quáaaa" thành "quáa"). Có thêm một bản **bỏ dấu** của câu, ghép vào sau câu gốc, để model khớp được cả câu gõ không dấu.

In [ ]:
import unicodedata

def strip_accents(s):
    s = s.replace("đ", "d").replace("Đ", "D")
    return "".join(c for c in unicodedata.normalize("NFD", s) if unicodedata.category(c) != "Mn")

def normalize(s):
    s = unicodedata.normalize("NFC", str(s)).lower()
    s = re.sub(r"(ord|tk|#)[-\w]*\d[\w-]*", " <orderid> ", s)
    s = re.sub(r"\d+([.,]\d+)*\s*(k|tr|đ|vnd|vnđ|d)?\b", " <num> ", s)
    s = re.sub(r"(.)\1{2,}", r"\1\1", s)
    return re.sub(r"\s+", " ", s).strip()

def featurize(s):
    n = normalize(s)
    return n + " || " + strip_accents(n)

for t in train_df.text.sample(5, random_state=1):
    print(t, "\n   ->", featurize(t))

## 4. Baseline: TF-IDF (word + char n-gram) và LinearSVC / Logistic Regression

In [ ]:
from sklearn.calibration import CalibratedClassifierCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.svm import LinearSVC

def make_pipe(clf):
    return Pipeline([
        ("prep", FunctionTransformer(lambda xs: [featurize(x) for x in xs])),
        ("feats", FeatureUnion([
            ("word", TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True, min_df=1)),
            ("char", TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), sublinear_tf=True, min_df=2)),
        ])),
        ("clf", clf),
    ])

cv = StratifiedKFold(5, shuffle=True, random_state=SEED)
X_tr, y_tr = train_df.text.tolist(), train_df.intent.tolist()
X_te, y_te = test_df.text.tolist(), test_df.intent.tolist()

search = {
    "svm": (make_pipe(LinearSVC(class_weight="balanced")), {"clf__C": [0.1, 0.3, 1, 3]}),
    "logreg": (make_pipe(LogisticRegression(max_iter=3000, class_weight="balanced")), {"clf__C": [3, 10, 30, 100]}),
}
results, best = {}, {}
for name, (pipe, grid) in search.items():
    t0 = time.time()
    gs = GridSearchCV(pipe, grid, cv=cv, scoring="f1_macro", n_jobs=-1).fit(X_tr, y_tr)
    pred = gs.predict(X_te)
    results[name] = {"cv_f1": gs.best_score_, "params": gs.best_params_,
                     "test_acc": accuracy_score(y_te, pred), "test_f1": f1_score(y_te, pred, average="macro")}
    best[name] = gs.best_estimator_
    print(f"{name:7s} {results[name]}  ({time.time()-t0:.0f}s)")

### Model chọn: LinearSVC đã hiệu chỉnh xác suất

Gateway chặn theo ngưỡng `confidence < 0.60`, nên confidence phải là **xác suất thật**. Vì vậy ta bọc LinearSVC bằng `CalibratedClassifierCV` thay vì dùng `sigmoid(margin)` như model cũ.

In [ ]:
C_best = results["svm"]["params"]["clf__C"]
final = make_pipe(CalibratedClassifierCV(LinearSVC(C=C_best, class_weight="balanced"), cv=5, method="sigmoid"))
final.fit(X_tr, y_tr)
proba = final.predict_proba(X_te)
pred = final.classes_[proba.argmax(1)]
conf = proba.max(1)
results["svm_calibrated"] = {"test_acc": accuracy_score(y_te, pred), "test_f1": f1_score(y_te, pred, average="macro")}
print(results["svm_calibrated"])
print(classification_report(y_te, pred, labels=LABELS, zero_division=0))

## 5. Phân tích lỗi

In [ ]:
cm = confusion_matrix(y_te, pred, labels=LABELS)
fig, ax = plt.subplots(figsize=(10, 9))
ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(len(LABELS)), LABELS, rotation=75); ax.set_yticks(range(len(LABELS)), LABELS)
for i in range(len(LABELS)):
    for j in range(len(LABELS)):
        if cm[i, j]: ax.text(j, i, cm[i, j], ha="center", va="center", color="white" if cm[i, j] > cm.max()/2 else "black")
ax.set_xlabel("Dự đoán"); ax.set_ylabel("Đúng"); plt.tight_layout(); plt.show()

test_df["pred"], test_df["conf"] = pred, conf
test_df["correct"] = test_df.pred == test_df.intent
print("Accuracy theo kiểu câu:")
display(test_df.groupby("style").correct.agg(["mean", "count"]).query("count >= 3").sort_values("mean"))
print("Các câu sai:")
display(test_df.loc[~test_df.correct, ["text", "intent", "pred", "conf"]])

### Ngưỡng confidence: bao nhiêu câu bị chuyển cho người, và độ chính xác của phần còn lại

Mô phỏng guardrail: câu có confidence dưới ngưỡng thì chuyển nhân viên (`ESCALATE_HUMAN`). Bảng dưới cho thấy với từng ngưỡng, bao nhiêu câu được giữ lại và độ chính xác trên số câu đó.

In [ ]:
rows = []
for th in np.arange(0.2, 0.95, 0.05):
    keep = test_df.conf >= th
    rows.append({"threshold": round(th, 2), "coverage": keep.mean(),
                 "acc_on_kept": test_df.correct[keep].mean() if keep.any() else np.nan})
thr = pd.DataFrame(rows); display(thr)
plt.plot(thr.threshold, thr.coverage, label="coverage"); plt.plot(thr.threshold, thr.acc_on_kept, label="accuracy (kept)")
plt.axvline(0.60, ls="--", c="gray"); plt.legend(); plt.xlabel("ngưỡng"); plt.show()

### So sánh với LLM (nếu đã upload `results_<model>.jsonl` từ `eval_llm.py`)

In [ ]:
for path in find("results_*.jsonl"):
    llm = pd.read_json(path, lines=True).set_index("id")["pred"]
    m = test_df.join(llm.rename("llm_pred"), on="id").dropna(subset=["llm_pred"])
    if len(m):
        name = os.path.basename(path).replace("results_", "").replace(".jsonl", "")
        results[f"llm_{name}"] = {"test_acc": accuracy_score(m.intent, m.llm_pred),
                                  "test_f1": f1_score(m.intent, m.llm_pred, average="macro"), "n": len(m)}
        print(name, results[f"llm_{name}"])
        print("  ML sai nhưng LLM đúng:", int(((m.pred != m.intent) & (m.llm_pred == m.intent)).sum()),
              "| LLM sai nhưng ML đúng:", int(((m.pred == m.intent) & (m.llm_pred != m.intent)).sum()))

## 6. (Tuỳ chọn, GPU) Fine-tune Transformer

Với 800 câu train, fine-tune thường hơn TF-IDF ở câu dài, câu gián tiếp và câu sát ranh giới nhãn. `vinai/phobert-base-v2` tốt cho tiếng Việt chuẩn. `xlm-roberta-base` chịu được tiếng Anh và câu trộn Việt–Anh tốt hơn.

Đặt `RUN_TRANSFORMER = True` để chạy. Bước này cần GPU và Internet.

In [ ]:
RUN_TRANSFORMER = False
MODEL_NAME = "xlm-roberta-base"   # hoặc "vinai/phobert-base-v2"
EPOCHS, LR, BS, MAXLEN = 12, 3e-5, 16, 96

In [ ]:
if RUN_TRANSFORMER:
    import torch
    from torch.utils.data import DataLoader, Dataset
    from transformers import AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup

    device = "cuda" if torch.cuda.is_available() else "cpu"
    torch.manual_seed(SEED)
    l2i = {l: i for i, l in enumerate(LABELS)}
    tok = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=len(LABELS), id2label=dict(enumerate(LABELS)), label2id=l2i).to(device)

    class DS(Dataset):
        def __init__(self, d): self.t, self.y = d.text.tolist(), [l2i[x] for x in d.intent]
        def __len__(self): return len(self.t)
        def __getitem__(self, i): return self.t[i], self.y[i]
    def collate(b):
        enc = tok([x for x, _ in b], padding=True, truncation=True, max_length=MAXLEN, return_tensors="pt")
        enc["labels"] = torch.tensor([y for _, y in b]); return enc

    # 10% train làm validation để chọn epoch, test giữ nguyên
    tr_df, va_df = train_test_split(train_df, test_size=0.1, stratify=train_df.intent, random_state=SEED)
    dl_tr = DataLoader(DS(tr_df), batch_size=BS, shuffle=True, collate_fn=collate)
    dl_va = DataLoader(DS(va_df), batch_size=64, collate_fn=collate)
    dl_te = DataLoader(DS(test_df), batch_size=64, collate_fn=collate)

    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    sched = get_linear_schedule_with_warmup(opt, int(0.1 * EPOCHS * len(dl_tr)), EPOCHS * len(dl_tr))

    def run_eval(dl):
        model.eval(); P = []
        with torch.no_grad():
            for b in dl:
                b = {k: v.to(device) for k, v in b.items()}
                P.append(torch.softmax(model(**b).logits, -1).cpu())
        return torch.cat(P).numpy()

    best_f1, best_state = -1, None
    for ep in range(EPOCHS):
        model.train(); tot = 0
        for b in dl_tr:
            b = {k: v.to(device) for k, v in b.items()}
            loss = model(**b).loss; loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); sched.step(); opt.zero_grad(); tot += loss.item()
        va_pred = [LABELS[i] for i in run_eval(dl_va).argmax(1)]
        f1 = f1_score(va_df.intent, va_pred, average="macro")
        print(f"epoch {ep+1}: loss {tot/len(dl_tr):.3f} | val macro-F1 {f1:.4f}")
        if f1 > best_f1:
            best_f1, best_state = f1, {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    model.load_state_dict(best_state)
    P = run_eval(dl_te)
    tf_pred = [LABELS[i] for i in P.argmax(1)]
    results["transformer"] = {"model": MODEL_NAME, "test_acc": accuracy_score(y_te, tf_pred),
                              "test_f1": f1_score(y_te, tf_pred, average="macro")}
    print(results["transformer"])
    print(classification_report(y_te, tf_pred, labels=LABELS, zero_division=0))
    test_df["transformer_pred"], test_df["transformer_conf"] = tf_pred, P.max(1)
    model.save_pretrained(f"{OUT_DIR}/intent_transformer"); tok.save_pretrained(f"{OUT_DIR}/intent_transformer")

## 7. Tổng kết và xuất model

In [ ]:
summary = pd.DataFrame(results).T[["test_acc", "test_f1"]].sort_values("test_f1", ascending=False)
display(summary)

# Fit lại model sklearn trên TOÀN BỘ 1000 câu để triển khai.
# Lưu ý: hàm tiền xử lý dùng lambda nên lưu bằng cloudpickle (joblib thường không pickle được lambda).
import cloudpickle
deploy = make_pipe(CalibratedClassifierCV(LinearSVC(C=C_best, class_weight="balanced"), cv=5, method="sigmoid"))
deploy.fit(df.text.tolist(), df.intent.tolist())
with open(f"{OUT_DIR}/intent_tfidf_svm.pkl", "wb") as f:
    cloudpickle.dump(deploy, f)

test_df.to_csv(f"{OUT_DIR}/test_predictions.csv", index=False)
with open(f"{OUT_DIR}/metrics.json", "w", encoding="utf-8") as f:
    json.dump({k: {kk: (float(vv) if isinstance(vv, (np.floating, float)) else vv) for kk, vv in v.items()}
               for k, v in results.items()}, f, ensure_ascii=False, indent=2, default=str)
print(os.listdir(OUT_DIR))

for q in ["đơn ORD-1001 tới đâu r shop", "toi bi tru tien 2 lan", "cho gặp nhân viên thật đi", "hôm nay trời đẹp ghê"]:
    p = deploy.predict_proba([q])[0]; print(f"{q:35s} -> {deploy.classes_[p.argmax()]} ({p.max():.2f})")

## Ghi chú khi đưa model vào `intent_service`

- `IntentPredictor` **ưu tiên** bộ 3 file trong `ML-model/`. Muốn dùng model mới thì xoá hoặc đổi tên thư mục đó, rồi đặt `intent_tfidf_svm.pkl` vào `MODEL_PATH`.
- Model được lưu bằng `cloudpickle`, nên `predictor.py` cần load bằng `cloudpickle.load` thay vì `joblib.load`, và môi trường phải có cùng phiên bản scikit-learn.
- `TARGET_INTENTS` cần coi `other_ecommerce` và `out_of_scope` là ngoài phạm vi. Hiện 2 nhãn này không có trong danh sách nên guardrail sẽ tự chuyển chúng cho nhân viên, đúng như mong muốn.
- 1000 câu vẫn là ít. Nên bổ sung dữ liệu thật từ log `copilot_telemetry.sqlite3`, đặc biệt các câu nhân viên đã sửa nhãn.